In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Given a 2D grid of 32-bit floating point values, apply one iteration of the 5-point Jacobi stencil:
  each interior cell of the output is set to the average of its four cardinal neighbors (top, bottom,
  left, right) from the input grid. Boundary cells (first/last row and column) are copied unchanged
  from the input to the output.
</p>

<svg width="320" height="280" viewBox="0 0 320 280" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;" font-family="monospace" font-size="11">
  <rect width="320" height="280" rx="8" fill="#222"/>
  <defs>
    <marker id="arrj" markerWidth="7" markerHeight="5" refX="7" refY="2.5" orient="auto">
      <polygon points="0 0, 7 2.5, 0 5" fill="#44aa66"/>
    </marker>
  </defs>

  <!-- Title -->
  <text x="160" y="18" text-anchor="middle" fill="#ccc" font-size="11">5-Point Jacobi Stencil</text>

  <!-- 5x5 grid, cell 44x34, origin (30, 28) -->
  <!-- Row 0 (boundary) -->
  <rect x="30"  y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="126" y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="174" y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="222" y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <!-- Row 1 -->
  <rect x="30"  y="66" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="66" width="44" height="34" rx="2" fill="#333" stroke="#555"/>
  <rect x="126" y="66" width="44" height="34" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="2"/>
  <rect x="174" y="66" width="44" height="34" rx="2" fill="#333" stroke="#555"/>
  <rect x="222" y="66" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <!-- Row 2 -->
  <rect x="30"  y="104" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="104" width="44" height="34" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="2"/>
  <rect x="126" y="104" width="44" height="34" rx="2" fill="#1e2d4d" stroke="#4477bb" stroke-width="2"/>
  <rect x="174" y="104" width="44" height="34" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="2"/>
  <rect x="222" y="104" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <!-- Row 3 -->
  <rect x="30"  y="142" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="142" width="44" height="34" rx="2" fill="#333" stroke="#555"/>
  <rect x="126" y="142" width="44" height="34" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="2"/>
  <rect x="174" y="142" width="44" height="34" rx="2" fill="#333" stroke="#555"/>
  <rect x="222" y="142" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <!-- Row 4 (boundary) -->
  <rect x="30"  y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="126" y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="174" y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="222" y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>

  <!-- Neighbor labels -->
  <text x="148" y="88" text-anchor="middle" font-weight="bold" fill="#44aa66" font-size="12">T</text>
  <text x="100" y="126" text-anchor="middle" font-weight="bold" fill="#44aa66" font-size="12">L</text>
  <text x="148" y="124" text-anchor="middle" fill="#4477bb" font-size="9">(2,2)</text>
  <text x="196" y="126" text-anchor="middle" font-weight="bold" fill="#44aa66" font-size="12">R</text>
  <text x="148" y="164" text-anchor="middle" font-weight="bold" fill="#44aa66" font-size="12">B</text>

  <!-- Arrows from neighbors to center -->
  <line x1="148" y1="100" x2="148" y2="107" stroke="#44aa66" stroke-width="1.5" marker-end="url(#arrj)"/>
  <line x1="122" y1="121" x2="129" y2="121" stroke="#44aa66" stroke-width="1.5" marker-end="url(#arrj)"/>
  <line x1="174" y1="121" x2="167" y2="121" stroke="#44aa66" stroke-width="1.5" marker-end="url(#arrj)"/>
  <line x1="148" y1="142" x2="148" y2="135" stroke="#44aa66" stroke-width="1.5" marker-end="url(#arrj)"/>

  <!-- Legend -->
  <rect x="30" y="228" width="14" height="14" rx="2" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="48" y="240" fill="#ccc" font-size="9">Center cell</text>
  <rect x="115" y="228" width="14" height="14" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="133" y="240" fill="#ccc" font-size="9">Neighbors</text>
  <rect x="200" y="228" width="14" height="14" rx="2" fill="#2a2a2a" stroke="#444" stroke-width="1.5"/>
  <text x="218" y="240" fill="#ccc" font-size="9">Boundary</text>

  <!-- Formula -->
  <text x="160" y="264" text-anchor="middle" fill="#ccc" font-size="11">output[i,j] = &#xbc; &#xd7; (top + bottom + left + right)</text>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>output</code></li>
  <li>Read exclusively from <code>input</code> and write exclusively to <code>output</code> (do not update <code>input</code>)</li>
</ul>

<h2>Example:</h2>
<p>
Input ($4 \times 4$):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0 \\
9.0 & 10.0 & 11.0 & 12.0 \\
13.0 & 14.0 & 15.0 & 16.0
\end{bmatrix}
$$
Output ($4 \times 4$):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0 \\
9.0 & 10.0 & 11.0 & 12.0 \\
13.0 & 14.0 & 15.0 & 16.0
\end{bmatrix}
$$
Interior cell $(1,1)$: $0.25 \times (\text{input}[0,1] + \text{input}[2,1] + \text{input}[1,0] + \text{input}[1,2])$
$= 0.25 \times (2.0 + 10.0 + 5.0 + 7.0) = 6.0$<br>
Interior cell $(1,2)$: $0.25 \times (\text{input}[0,2] + \text{input}[2,2] + \text{input}[1,1] + \text{input}[1,3])$
$= 0.25 \times (3.0 + 11.0 + 6.0 + 8.0) = 7.0$<br>
Interior cell $(2,1)$: $0.25 \times (\text{input}[1,1] + \text{input}[3,1] + \text{input}[2,0] + \text{input}[2,2])$
$= 0.25 \times (6.0 + 14.0 + 9.0 + 11.0) = 10.0$<br>
Interior cell $(2,2)$: $0.25 \times (\text{input}[1,2] + \text{input}[3,2] + \text{input}[2,1] + \text{input}[2,3])$
$= 0.25 \times (7.0 + 15.0 + 10.0 + 12.0) = 11.0$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>rows</code>, <code>cols</code> &le; 16,384</li>
  <li>Input values are in the range [-100, 100]</li>
  <li>All values are 32-bit floats</li>
  <li>Performance is measured with <code>rows</code> = 8,192, <code>cols</code> = 8,192</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// input, output are device pointers
extern "C" void solve(const float* input, float* output, int rows, int cols) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# input, output are tensors on the GPU
@cute.jit
def solve(input: cute.Tensor, output: cute.Tensor, rows: cute.Int32, cols: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# input is a tensor on the GPU
@jax.jit
def solve(input: jax.Array, rows: int, cols: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# input, output are device pointers
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    rows: Int32,
    cols: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, rows: int, cols: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, rows: int, cols: int):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="2D Jacobi Stencil",
            atol=1e-05,
            rtol=1e-05,
            num_gpus=1,
            access_tier="free",
        )

    def reference_impl(
        self,
        input: torch.Tensor,
        output: torch.Tensor,
        rows: int,
        cols: int,
    ):
        assert input.shape == (rows, cols)
        assert output.shape == (rows, cols)
        assert input.dtype == torch.float32
        assert input.device.type == "cuda"

        # Copy boundary cells unchanged
        output.copy_(input)

        # Apply 5-point stencil to interior cells:
        # output[i, j] = 0.25 * (input[i-1,j] + input[i+1,j] + input[i,j-1] + input[i,j+1])
        output[1:-1, 1:-1] = 0.25 * (
            input[0:-2, 1:-1]  # top neighbor
            + input[2:, 1:-1]  # bottom neighbor
            + input[1:-1, 0:-2]  # left neighbor
            + input[1:-1, 2:]  # right neighbor
        )

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "input": (ctypes.POINTER(ctypes.c_float), "in"),
            "output": (ctypes.POINTER(ctypes.c_float), "out"),
            "rows": (ctypes.c_int, "in"),
            "cols": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        input = torch.tensor(
            [
                [1.0, 2.0, 3.0, 4.0],
                [5.0, 6.0, 7.0, 8.0],
                [9.0, 10.0, 11.0, 12.0],
                [13.0, 14.0, 15.0, 16.0],
            ],
            device="cuda",
            dtype=dtype,
        )
        output = torch.empty((4, 4), device="cuda", dtype=dtype)
        return {
            "input": input,
            "output": output,
            "rows": 4,
            "cols": 4,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype = torch.float32
        tests = []

        # minimal_3x3 (only one interior cell)
        tests.append(
            {
                "input": torch.tensor(
                    [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]],
                    device="cuda",
                    dtype=dtype,
                ),
                "output": torch.empty((3, 3), device="cuda", dtype=dtype),
                "rows": 3,
                "cols": 3,
            }
        )

        # minimal_1x1 (all boundary, no interior cells)
        tests.append(
            {
                "input": torch.tensor([[42.0]], device="cuda", dtype=dtype),
                "output": torch.empty((1, 1), device="cuda", dtype=dtype),
                "rows": 1,
                "cols": 1,
            }
        )

        # single_row (all boundary)
        tests.append(
            {
                "input": torch.tensor([[1.0, 2.0, 3.0, 4.0]], device="cuda", dtype=dtype),
                "output": torch.empty((1, 4), device="cuda", dtype=dtype),
                "rows": 1,
                "cols": 4,
            }
        )

        # single_col (all boundary)
        tests.append(
            {
                "input": torch.tensor([[1.0], [2.0], [3.0], [4.0]], device="cuda", dtype=dtype),
                "output": torch.empty((4, 1), device="cuda", dtype=dtype),
                "rows": 4,
                "cols": 1,
            }
        )

        # all_zeros (interior should stay zero)
        tests.append(
            {
                "input": torch.zeros((16, 16), device="cuda", dtype=dtype),
                "output": torch.empty((16, 16), device="cuda", dtype=dtype),
                "rows": 16,
                "cols": 16,
            }
        )

        # uniform_constant (interior stays the same when all values equal)
        tests.append(
            {
                "input": torch.full((32, 32), 3.14, device="cuda", dtype=dtype),
                "output": torch.empty((32, 32), device="cuda", dtype=dtype),
                "rows": 32,
                "cols": 32,
            }
        )

        # power_of_2_square_64
        tests.append(
            {
                "input": torch.empty((64, 64), device="cuda", dtype=dtype).uniform_(-5.0, 5.0),
                "output": torch.empty((64, 64), device="cuda", dtype=dtype),
                "rows": 64,
                "cols": 64,
            }
        )

        # power_of_2_square_128
        tests.append(
            {
                "input": torch.empty((128, 128), device="cuda", dtype=dtype).uniform_(-10.0, 10.0),
                "output": torch.empty((128, 128), device="cuda", dtype=dtype),
                "rows": 128,
                "cols": 128,
            }
        )

        # non_power_of_2_30x30
        tests.append(
            {
                "input": torch.empty((30, 30), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "output": torch.empty((30, 30), device="cuda", dtype=dtype),
                "rows": 30,
                "cols": 30,
            }
        )

        # non_power_of_2_100x100
        tests.append(
            {
                "input": torch.empty((100, 100), device="cuda", dtype=dtype).uniform_(-3.0, 3.0),
                "output": torch.empty((100, 100), device="cuda", dtype=dtype),
                "rows": 100,
                "cols": 100,
            }
        )

        # non_square_255x33
        tests.append(
            {
                "input": torch.empty((255, 33), device="cuda", dtype=dtype).uniform_(-2.0, 2.0),
                "output": torch.empty((255, 33), device="cuda", dtype=dtype),
                "rows": 255,
                "cols": 33,
            }
        )

        # negative_values_non_square_17x97
        tests.append(
            {
                "input": torch.empty((17, 97), device="cuda", dtype=dtype).uniform_(-100.0, 0.0),
                "output": torch.empty((17, 97), device="cuda", dtype=dtype),
                "rows": 17,
                "cols": 97,
            }
        )

        # realistic_medium_512x256
        tests.append(
            {
                "input": torch.empty((512, 256), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
                "output": torch.empty((512, 256), device="cuda", dtype=dtype),
                "rows": 512,
                "cols": 256,
            }
        )

        # realistic_large_1024x1024
        tests.append(
            {
                "input": torch.empty((1024, 1024), device="cuda", dtype=dtype).uniform_(-5.0, 5.0),
                "output": torch.empty((1024, 1024), device="cuda", dtype=dtype),
                "rows": 1024,
                "cols": 1024,
            }
        )

        return tests

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype = torch.float32
        rows = 8192
        cols = 8192
        return {
            "input": torch.empty((rows, cols), device="cuda", dtype=dtype).uniform_(-1.0, 1.0),
            "output": torch.empty((rows, cols), device="cuda", dtype=dtype),
            "rows": rows,
            "cols": cols,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
